In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict

print("Starting the ultimate pipeline... grab a chai.")

# 0. Output folder setup
os.makedirs("../data/model_ready/", exist_ok=True)

# 1. LOAD THE TABULAR DATA
df_train = pd.read_csv("../data/train.csv")
df_valid = pd.read_csv("../data/valid.csv")
df_test = pd.read_csv("../data/test.csv")

global_mean = df_train['SalaryNormalized'].mean()

# ==========================================
# 2. THE GOD-TIER HIERARCHICAL LOCATION ENCODER
# ==========================================
print("Parsing the Location Tree like a boss...")

# A. Read the tree into a dictionary: leaf -> full path
# e.g. 'Shadwell': ['UK', 'London', 'East London', 'Shadwell']
tree_paths = {}
with open('../data/Location_tree.csv', 'r', encoding='utf-8') as f:
    for line in f:
        # Clean up the line and split by ~
        path = line.strip().replace('"', '').split('~')
        leaf = path[-1]
        tree_paths[leaf] = path 

# B. Calculate the mean salary for EVERY region level using training data
print("Calculating geographical wealth...")
node_salaries = defaultdict(list)

# We zip through the training data to assign salaries to all parent regions
for loc, salary in zip(df_train['LocationNormalized'], df_train['SalaryNormalized']):
    if loc in tree_paths:
        for region in tree_paths[loc]:
            node_salaries[region].append(salary)
    else:
        # If the location isn't in the tree at all, just track it normally
        node_salaries[loc].append(salary)

# Average them out
node_means = {node: np.mean(sals) for node, sals in node_salaries.items()}

# C. The Fallback Function
def encode_location(loc):
    """Tries specific town -> parent region -> grandparent region -> global mean"""
    if loc in tree_paths:
        # Walk backwards: Shadwell -> East London -> London -> UK
        for region in reversed(tree_paths[loc]):
            if region in node_means:
                return node_means[region]
    
    # If not in tree, but was in train data anyway
    if loc in node_means:
        return node_means[loc]
        
    return global_mean

print("Applying hierarchical location encoding...")
df_train['LocationNormalized'] = df_train['LocationNormalized'].apply(encode_location)
df_valid['LocationNormalized'] = df_valid['LocationNormalized'].apply(encode_location)
df_test['LocationNormalized']  = df_test['LocationNormalized'].apply(encode_location)

# ==========================================
# 3. NORMAL TARGET ENCODING (For the rest)
# ==========================================
print("Target encoding Company, Category, and SourceName...")
target_cols = ['Company', 'Category', 'SourceName']

for col in target_cols:
    col_means = df_train.groupby(col)['SalaryNormalized'].mean()
    df_train[col] = df_train[col].map(col_means).fillna(global_mean)
    df_valid[col] = df_valid[col].map(col_means).fillna(global_mean)
    df_test[col] = df_test[col].map(col_means).fillna(global_mean)

# ==========================================
# 4. DROP THE TRASH
# ==========================================
drop_cols = ['Id', 'Title', 'FullDescription', 'LocationRaw']
print("Taking out the garbage...")
for df in [df_train, df_valid, df_test]:
    df.drop(columns=drop_cols, errors='ignore', inplace=True)

# ==========================================
# 5. ONE-HOT ENCODING
# ==========================================
ohe_cols = ['ContractType', 'ContractTime']
print("One-Hot Encoding...")
df_train = pd.get_dummies(df_train, columns=ohe_cols, dummy_na=True)
df_valid = pd.get_dummies(df_valid, columns=ohe_cols, dummy_na=True)
df_test = pd.get_dummies(df_test, columns=ohe_cols, dummy_na=True)

df_train, df_valid = df_train.align(df_valid, join='left', axis=1, fill_value=0)
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

# ==========================================
# 6. LOAD THE DISTILBERT EMBEDDINGS
# ==========================================
print("Loading the DistilBERT brain juice...")
train_emb = np.load("../data/train_embeds.npy")
valid_emb = np.load("../data/valid_embeds.npy")
test_emb = np.load("../data/test_embeds.npy")

emb_cols = [f"bert_{i}" for i in range(train_emb.shape[1])]
df_train_emb = pd.DataFrame(train_emb, index=df_train.index, columns=emb_cols)
df_valid_emb = pd.DataFrame(valid_emb, index=df_valid.index, columns=emb_cols)
df_test_emb = pd.DataFrame(test_emb, index=df_test.index, columns=emb_cols)

# ==========================================
# 7. THE GRAND STITCHING
# ==========================================
print("Stitching it all together...")
final_train = pd.concat([df_train, df_train_emb], axis=1)
final_valid = pd.concat([df_valid, df_valid_emb], axis=1)
final_test = pd.concat([df_test, df_test_emb], axis=1)

# ==========================================
# 8. SAVE AS PARQUET
# ==========================================
print("Saving to ../data/model_ready/ as Parquet...")
final_train.to_parquet("../data/model_ready/train_final.parquet", index=False)
final_valid.to_parquet("../data/model_ready/valid_final.parquet", index=False)
final_test.to_parquet("../data/model_ready/test_final.parquet", index=False)

print(f"Done! Final Train Shape: {final_train.shape}")
print("Your hierarchy logic is flawless. Go run this and nuke the competition.")

Starting the ultimate pipeline... grab a chai.
Parsing the Location Tree like a boss...
Calculating geographical wealth...
Applying hierarchical location encoding...
Target encoding Company, Category, and SourceName...
Taking out the garbage...
One-Hot Encoding...
Loading the DistilBERT brain juice...
Stitching it all together...
Saving to ../data/model_ready/ as Parquet...
Done! Final Train Shape: (244768, 779)
Your hierarchy logic is flawless. Go run this and nuke the competition.
